# Step 3 — VLM Caption Overlay (Ollama)

This notebook runs an Ollama VLM to generate scene captions and overlays them on annotated videos from Step 2 (YOLO + CLIP) or Step 1 (YOLO-only).

Prerequisites:

- `ollama` installed and running (`ollama serve`),
- Model pulled: `ollama pull qwen3-vl:4b`,
- Python client installed in the notebook environment: `pip install ollama`.

Inputs:

- Default input root: `runs_output/detect/`.
- Auto-discovery prefers `clip_predict`, then the highest `predictN` (Ultralytics default; `predict-N` / `predict_N` also accepted), then `predict`, then any other direct child folder under `runs_output/detect/` that already contains a video.
- You can override with `PREDICT_DIR` (full folder path) or `INPUT_ROOT` (root containing the output folders).
- You can override the exact video with `INPUT_VIDEO` (full file path).

Captioning:

- Default mode is **adaptive** (caption only when the scene changes),
- A minimum gap of **5 seconds** is enforced so captions are readable.

Run order (beginner-friendly):

1. Run the install cell.
2. Run import/setup cells (defines prompts and helper functions).
3. Run the video-generation cell only if you want to create or refresh output video.
4. Run the chat cell to ask questions about one selected frame.

What you learn here:

- How to convert OpenCV frames to image bytes for a VLM request,
- How to choose frames by timestamp using FPS/frame index,
- How to combine CV outputs (YOLO boxes and optional CLIP labels) with language outputs (scene Q&A).

In [2]:
!pip install -q ollama

In [4]:
# VLM caption overlay (Ollama) - Step 3
import io
import os
import glob
import textwrap
import cv2
import re
from PIL import Image

In [ ]:
try:
    from ollama import chat
except ImportError as exc:
    raise ImportError("Missing ollama Python client. Install it in your environment: pip install ollama") from exc

# Configuration (fixed model for this project)
VLM_MODEL = "qwen3-vl:4b"
VLM_PROMPT = "Describe the driving scene in one short sentence."
CHAT_PROMPT = (
    "You are a skilled dashcam scene analyst. Use only what is visible in this single frame. "
    "Answer the user's question first in clear natural language. "
    "Be precise, practical, and evidence-based.\n\n"
    "Guidelines:\n"
    "- If the user asks for car models/brands, provide only cautious possibilities (if any) and include confidence (high/medium/low).\n"
    "- Prefer reliable attributes first: vehicle type, color, lane position, relative distance, motion cue.\n"
    "- If exact model/brand is not visually verifiable, state that explicitly.\n"
    "- Do not invent text, objects, or events not visible.\n"
    "- Keep the answer concise but informative.\n\n"
    "Optional structure when helpful:\n"
    "Answer: <direct answer>\n"
    "Evidence: <2-5 short bullets from the frame>\n"
    "Confidence: <high/medium/low with reason>"
 )
CHAT_MAX_WORDS = int(os.getenv("CHAT_MAX_WORDS", "180"))
CHAT_MAX_TOKENS = int(os.getenv("CHAT_MAX_TOKENS", "320"))
PREDICT_DIR = os.getenv("PREDICT_DIR", "")
INPUT_ROOT = os.getenv("INPUT_ROOT", os.path.join(os.getcwd(), "runs_output", "detect"))
INPUT_VIDEO = os.getenv("INPUT_VIDEO", "")
VLM_MAX_ERRORS = int(os.getenv("VLM_MAX_ERRORS", "5"))

# If you move to a cloud VLM later, do this:
# 1) Set a host or API base URL in your terminal (example):
#    export OLLAMA_HOST="http://<host>:<port>"
# 2) If the provider needs an API key, set it in the terminal:
#    export VLM_API_KEY="<your_key>"
# 3) Update the client call in get_caption() to use that provider's SDK.

# Captioning controls
CAPTION_MODE = os.getenv("CAPTION_MODE", "adaptive")  # adaptive = scene change, fixed = every N seconds
CAPTION_EVERY_SEC = float(os.getenv("CAPTION_EVERY_SEC", "8.0"))  # used only in fixed mode
CAPTION_MIN_SEC = float(os.getenv("CAPTION_MIN_SEC", "5.0"))  # minimum gap between captions
CAPTION_DIFF_THRESHOLD = float(os.getenv("CAPTION_DIFF_THRESHOLD", "12.0"))  # higher = fewer updates


def frame_to_bytes(frame_bgr):
    # Convert OpenCV BGR frame to JPEG bytes for Ollama
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return buf.getvalue()


def get_caption(frame_bgr):
    # Send one frame to the VLM and get a single-sentence caption
    # If you switch providers, replace chat(...) below with that SDK call.
    img_bytes = frame_to_bytes(frame_bgr)
    response = chat(
        model=VLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": VLM_PROMPT,
                "images": [img_bytes],
            }
        ],
        stream=False,
    )
    return response.message.content.strip().replace("\n", " ")


def overlay_caption(frame_bgr, caption):
    # Draw the caption on a black banner for readability
    if not caption:
        return frame_bgr
    lines = textwrap.wrap(caption, width=40)
    font_scale = 3.0
    line_height = 120
    pad = 50
    thickness = 10
    box_height = pad * 2 + line_height * len(lines)
    cv2.rectangle(frame_bgr, (0, 0), (frame_bgr.shape[1], box_height), (0, 0, 0), -1)
    y = pad + line_height
    for line in lines:
        cv2.putText(
            frame_bgr,
            line,
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )
        y += line_height
    return frame_bgr


def find_video_files(directory):
    return sorted(
        glob.glob(os.path.join(directory, "*.avi"))
        + glob.glob(os.path.join(directory, "*.mp4"))
        + glob.glob(os.path.join(directory, "*.mov"))
        + glob.glob(os.path.join(directory, "*.mkv")),
        key=os.path.getmtime,
    )


def candidate_output_dirs(input_root):
    if not os.path.isdir(input_root):
        raise FileNotFoundError(f"INPUT_ROOT not found: {input_root}")

    ordered_dirs = []
    seen = set()

    clip_predict = os.path.join(input_root, "clip_predict")
    if os.path.isdir(clip_predict):
        ordered_dirs.append(clip_predict)
        seen.add(clip_predict)

    numbered_predicts = []
    for path in glob.glob(os.path.join(input_root, "predict*")):
        if not os.path.isdir(path):
            continue
        name = os.path.basename(path)
        # Ultralytics defaults to predict2, predict3, ... (no separator);
        # also accept predict-2 / predict_2 just in case.
        match = re.fullmatch(r"predict[-_]?(\d+)", name)
        if match:
            numbered_predicts.append((int(match.group(1)), path))

    for _, path in sorted(numbered_predicts, key=lambda item: item[0], reverse=True):
        if path not in seen:
            ordered_dirs.append(path)
            seen.add(path)

    predict_dir = os.path.join(input_root, "predict")
    if os.path.isdir(predict_dir) and predict_dir not in seen:
        ordered_dirs.append(predict_dir)
        seen.add(predict_dir)

    for path in sorted(glob.glob(os.path.join(input_root, "*"))):
        if os.path.isdir(path) and path not in seen:
            ordered_dirs.append(path)
            seen.add(path)

    return ordered_dirs


def select_output_dir(input_root):
    for path in candidate_output_dirs(input_root):
        if find_video_files(path):
            return path
    return ""


def resolve_input_video():
    # Find the newest annotated video under runs_output/detect if none is specified
    if INPUT_VIDEO:
        if not os.path.isfile(INPUT_VIDEO):
            raise FileNotFoundError(f"INPUT_VIDEO not found: {INPUT_VIDEO}")
        return INPUT_VIDEO

    if PREDICT_DIR:
        if not os.path.isdir(PREDICT_DIR):
            raise FileNotFoundError(f"PREDICT_DIR not found: {PREDICT_DIR}")
        candidates = find_video_files(PREDICT_DIR)
        if not candidates:
            raise FileNotFoundError(f"No video files found in {PREDICT_DIR}")
        return candidates[-1]

    output_dir = select_output_dir(INPUT_ROOT)
    if not output_dir:
        raise FileNotFoundError(
            f"No annotated videos found under {INPUT_ROOT}. Set INPUT_ROOT, PREDICT_DIR, or INPUT_VIDEO."
        )

    return find_video_files(output_dir)[-1]


def should_caption_fixed(frame_idx, interval_frames):
    return frame_idx % interval_frames == 0


def should_caption_adaptive(frame_bgr, last_frame_bgr, elapsed_sec):
    # Update only if enough time passed AND the scene changed enough
    if last_frame_bgr is None:
        return True
    if elapsed_sec < CAPTION_MIN_SEC:
        return False
    gray_now = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    gray_last = cv2.cvtColor(last_frame_bgr, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(gray_now, gray_last)
    mean_diff = float(diff.mean())
    return mean_diff >= CAPTION_DIFF_THRESHOLD

### Run to generate the captioned video
Only run this if you want to reprocess the video. If you already have output files, skip to the chat cell.

In [ ]:
# Generate captioned video (skip if output already exists)
input_video = resolve_input_video()
base_name = os.path.splitext(os.path.basename(input_video))[0]
out_dir = os.path.join(os.path.dirname(input_video), "vlm_overlay")
os.makedirs(out_dir, exist_ok=True)
output_video = os.path.join(out_dir, f"{base_name}_vlm.mp4")

# Safety flag for beginners: keep existing result unless you explicitly re-run generation
SKIP_IF_OUTPUT_EXISTS = True
if SKIP_IF_OUTPUT_EXISTS and os.path.isfile(output_video):
    print(f"Skip generation (output already exists): {output_video}")
else:
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {input_video}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    interval_frames = max(1, int(round(fps * CAPTION_EVERY_SEC)))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    frame_idx = 0
    current_caption = ""
    last_caption_frame = None
    last_caption_idx = -999999
    error_count = 0
    completed = False
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            elapsed_sec = (frame_idx - last_caption_idx) / fps
            if CAPTION_MODE == "adaptive":
                do_caption = should_caption_adaptive(frame, last_caption_frame, elapsed_sec)
            else:
                do_caption = should_caption_fixed(frame_idx, interval_frames)
            if do_caption:
                try:
                    current_caption = get_caption(frame)
                    last_caption_frame = frame.copy()
                    last_caption_idx = frame_idx
                except Exception as exc:
                    error_count += 1
                    print(f"VLM error at frame {frame_idx}: {exc} (errors={error_count})")
                    if VLM_MAX_ERRORS > 0 and error_count >= VLM_MAX_ERRORS:
                        raise RuntimeError(f"VLM failed {error_count} times; aborting.")
            frame = overlay_caption(frame, current_caption)
            writer.write(frame)
            frame_idx += 1
        completed = True
    finally:
        cap.release()
        writer.release()
        # If the loop did not finish cleanly, remove the half-written output so the
        # next run is not silently skipped by SKIP_IF_OUTPUT_EXISTS on a corrupt file.
        if not completed and os.path.isfile(output_video):
            try:
                os.remove(output_video)
                print(f"Removed partial output: {output_video}")
            except OSError as rm_exc:
                print(f"Warning: could not remove partial output {output_video}: {rm_exc}")

    print(f"Input:  {input_video}")
    print(f"Output: {output_video}")

Input:  /home/dongyuan/Desktop/computer_vision/runs_output/detect/clip_predict/annotated_video.mp4
Output: /home/dongyuan/Desktop/computer_vision/runs_output/detect/clip_predict/vlm_overlay/annotated_video_vlm.mp4


# Quick chat about the video using a single frame at a chosen timestamp


In [ ]:
import os
import io
import time
import cv2
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Clear previous outputs from this cell to avoid stacked results
clear_output(wait=True)

missing = [
    name for name in ("chat", "VLM_MODEL", "VLM_PROMPT", "CHAT_PROMPT", "CHAT_MAX_WORDS", "CHAT_MAX_TOKENS")
    if name not in globals()
 ]
if missing:
    raise RuntimeError("Missing Step 3 setup: " + ", ".join(missing) + ". Run the Step 3 setup cell first.")

if "frame_to_bytes" not in globals():
    def frame_to_bytes(frame_bgr):
        # Local fallback if the Step 3 cell has not defined it yet
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        return buf.getvalue()

def get_frame_at_sec(video_path, sec):
    # Frame-index seeking is usually more reliable than millisecond seeking for MP4 files
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    target_idx = int(round(max(0.0, float(sec)) * fps))
    if frame_count:
        target_idx = min(target_idx, int(frame_count) - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Failed to read frame at {sec:.1f}s")
    actual_sec = target_idx / fps if fps else 0.0
    return frame, target_idx, actual_sec

def ask_vlm_about_video(question, sec):
    frame, frame_idx, actual_sec = get_frame_at_sec(video_path, sec)
    img_bytes = frame_to_bytes(frame)
    compact_instruction = (
        f"{CHAT_PROMPT}\n\n"
        f"User question: {question}\n"
        f"Target length: about {CHAT_MAX_WORDS} words maximum."
    )
    response = chat(
        model=VLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": compact_instruction,
                "images": [img_bytes],
            }
        ],
        options={
            "temperature": 0.2,
            "num_predict": CHAT_MAX_TOKENS,
        },
        stream=False,
    )
    answer = (response.message.content or "").strip()

    # Fallback retry with a simpler prompt if the model returned empty output
    if not answer:
        fallback_instruction = (
            "Answer the question directly from this frame in 2-5 concise sentences. "
            "Mention uncertainty where needed. "
            f"User question: {question}"
        )
        retry = chat(
            model=VLM_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": fallback_instruction,
                    "images": [img_bytes],
                }
            ],
            options={
                "temperature": 0.2,
                "num_predict": max(220, CHAT_MAX_TOKENS),
            },
            stream=False,
        )
        answer = (retry.message.content or "").strip()

    if not answer:
        answer = "No text returned by model for this frame. Try another timestamp."
    return answer, frame_idx, actual_sec

# Resolve video path and duration for the slider
if "compressed_video" in globals() and os.path.isfile(compressed_video):
    video_path = compressed_video
elif "output_video" in globals() and os.path.isfile(output_video):
    video_path = output_video
elif "input_video" in globals() and os.path.isfile(input_video):
    video_path = input_video
else:
    video_path = resolve_input_video()

cap = cv2.VideoCapture(video_path)
fps_local = cap.get(cv2.CAP_PROP_FPS) or 30.0
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
duration_sec = frame_count / fps_local if frame_count else 0
cap.release()
max_sec = max(1, int(duration_sec))

question_box = widgets.Text(
    value="",
    placeholder="Ask about the driving scene (e.g., 'Any pedestrians near the crosswalk?')",
    description="Question:",
    layout=widgets.Layout(width="100%"),
)
time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=max_sec,
    step=1,
    description="Time (s):",
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
)
ask_button = widgets.Button(description="Ask VLM", button_style="primary")
status_label = widgets.HTML(value="")
output_box = widgets.Output()

# Ensure only one live Q&A widget instance exists
if "_chat_ui" in globals():
    try:
        _chat_ui.close()
    except Exception:
        pass

# Guard against queued/double clicks when responses are slow
run_state = {"busy": False, "last_click_ts": 0.0}

def on_ask_clicked(_):
    now = time.time()
    if run_state["busy"] or (now - run_state["last_click_ts"] < 0.8):
        return
    run_state["busy"] = True
    run_state["last_click_ts"] = now

    ask_button.disabled = True
    status_label.value = "<b>Working...</b>"
    output_box.clear_output(wait=True)
    q = question_box.value.strip()
    if not q:
        with output_box:
            display(Markdown("Please type a question first."))
        status_label.value = ""
        ask_button.disabled = False
        run_state["busy"] = False
        return
    try:
        start_time = time.time()
        answer, frame_idx, actual_sec = ask_vlm_about_video(q, time_slider.value)
        elapsed = time.time() - start_time
        result_md = (
            "**Answer**\n\n"
            f"{answer}\n\n"
            "---\n"
            f"**Requested:** {time_slider.value}s\n"
            f"**Actual frame:** {frame_idx} ({actual_sec:.2f}s)\n"
            f"**Video:** {video_path}\n"
            f"**Latency:** {elapsed:.1f}s"
        )
        with output_box:
            display(Markdown(result_md))
        status_label.value = "<b>Done.</b>"
    except Exception as exc:
        with output_box:
            display(Markdown(f"**Error:** {exc}"))
        status_label.value = "<b>Error.</b>"
    finally:
        ask_button.disabled = False
        run_state["busy"] = False

# Use public API for handler replacement
if "_chat_on_click_handler" in globals():
    try:
        ask_button.on_click(_chat_on_click_handler, remove=True)
    except Exception:
        pass
_chat_on_click_handler = on_ask_clicked
ask_button.on_click(_chat_on_click_handler)

_chat_ui = widgets.VBox([question_box, time_slider, ask_button, status_label, output_box])
display(_chat_ui)

Next steps:

- Optionally add `try/except` around `chat()` to handle `ollama.ResponseError` and connection errors.
- If you want host control, set `OLLAMA_HOST` or provide a `Client(host=...)` from the `ollama` library.